In [ ]:
# Imports and Spark init
from pyspark.sql import SparkSession, functions as F
import pandas as pd
import matplotlib.pyplot as plt

spark = SparkSession.builder.appName("NutrientDensityExplorer").getOrCreate()
print("Spark version:", spark.version)

In [ ]:
# Configuration
input_path = "output/nutritional_profiles"  # Parquet directory
output_base = "output/NutrientDensityExplorer"
top_k = 20
order_desc = True  # True: highest densities first

# Nutrients to consider. Availability depends on columns present in the dataset.
nutrients = ["protein", "fiber", "sugar", "fat", "sodium"]

# Optional WWEIA category filtering: use exact matches of category column if present.
include_categories = []  # e.g., ["Milk and Dairy Products", "Poultry"]
exclude_categories = []  # e.g., ["Sugars, Sweets"]

# Display settings
display_metric = "protein"  # one of nutrients list
save_outputs = True

In [ ]:
# Read dataset and detect columns
df = spark.read.parquet(input_path)

def pick(df, candidates):
    # First exact match (case-insensitive), then contains
    for c in candidates:
        for col in df.columns:
            if col.lower() == c.lower():
                return col
    for c in candidates:
        for col in df.columns:
            if c.lower() in col.lower():
                return col
    return None

col_id = pick(df, ["fdc_id", "id"])
col_name = pick(df, ["description", "food_name", "name", "brand_name"])
col_kcal = pick(df, ["energy_kcal", "energy", "kcal", "calories", "1008"])
protein_col = pick(df, ["protein_g", "protein", "1003"])
fiber_col = pick(df, ["fiber_g", "fiber", "1079"])
carb_col = pick(df, ["carb_g", "carbohydrate", "carbohydrates", "1005"])
sugar_col = pick(df, ["sugar_g", "sugars", "sugar", "2000"])
fat_col = pick(df, ["fat_g", "fat", "total_fat", "1004"])
sodium_mg_col = pick(df, ["sodium_mg", "sodium", "1093"])
weight_g_col = pick(df, ["serving_size_g", "gram_weight", "weight_g", "portion_weight_g"])
cat_col = pick(df, ["wweia_category", "wweia_description", "category", "food_category", "wweia_food_category_description"])

print("Detected columns:")
print({
    'id': col_id,
    'name': col_name,
    'kcal': col_kcal,
    'protein_g': protein_col,
    'fiber_g': fiber_col,
    'carb_g': carb_col,
    'sugar_g': sugar_col,
    'fat_g': fat_col,
    'sodium_mg': sodium_mg_col,
    'weight_g': weight_g_col,
    'category': cat_col
})

In [ ]:
# Build densities per kcal and per 100g
df_base = df
if col_kcal:
    df_base = df_base.filter(F.col(col_kcal) > 0)

def per100g_expr(nut_col):
    # If weight column exists, normalize to 100g; else assume nut_col already per 100g
    if weight_g_col:
        return (F.col(nut_col) / F.col(weight_g_col)) * F.lit(100.0)
    else:
        return F.col(nut_col)

densities = []  # (metric_name, column_name)

if protein_col:
    df_base = df_base.withColumn("protein_per_kcal", F.col(protein_col) / F.col(col_kcal))
    df_base = df_base.withColumn("protein_per_100g", per100g_expr(protein_col))
    densities.append(("protein", "protein_per_kcal", "protein_per_100g"))
if fiber_col:
    df_base = df_base.withColumn("fiber_per_kcal", F.col(fiber_col) / F.col(col_kcal))
    df_base = df_base.withColumn("fiber_per_100g", per100g_expr(fiber_col))
    densities.append(("fiber", "fiber_per_kcal", "fiber_per_100g"))
if sugar_col:
    df_base = df_base.withColumn("sugar_per_kcal", F.col(sugar_col) / F.col(col_kcal))
    df_base = df_base.withColumn("sugar_per_100g", per100g_expr(sugar_col))
    densities.append(("sugar", "sugar_per_kcal", "sugar_per_100g"))
if fat_col:
    df_base = df_base.withColumn("fat_per_kcal", F.col(fat_col) / F.col(col_kcal))
    df_base = df_base.withColumn("fat_per_100g", per100g_expr(fat_col))
    densities.append(("fat", "fat_per_kcal", "fat_per_100g"))
if sodium_mg_col:
    df_base = df_base.withColumn("sodium_mg_per_kcal", F.col(sodium_mg_col) / F.col(col_kcal))
    df_base = df_base.withColumn("sodium_mg_per_100g", per100g_expr(sodium_mg_col))
    densities.append(("sodium", "sodium_mg_per_kcal", "sodium_mg_per_100g"))

print("Computed density columns for:", [d[0] for d in densities])

In [ ]:
# Optional category filtering (only if category column is available)
df_filt = df_base
if cat_col:
    if include_categories:
        lowers = [c.lower() for c in include_categories]
        df_filt = df_filt.filter(F.lower(F.col(cat_col)).isin(lowers))
    if exclude_categories:
        lowers_ex = [c.lower() for c in exclude_categories]
        df_filt = df_filt.filter(~F.lower(F.col(cat_col)).isin(lowers_ex))
else:
    print("No category column detected; category filters will be ignored.")

print("Rows after filtering:", df_filt.count())

In [ ]:
# Helpers: select columns and compute Top-K
def select_cols(metric_col):
    cols = []
    for c in [col_id, col_name, col_kcal, cat_col]:
        if c:
            cols.append(c)
    cols.append(metric_col)
    return cols

def topk(df_in, metric_col, desc=True):
    order_col = F.col(metric_col).desc() if desc else F.col(metric_col).asc()
    return df_in.orderBy(order_col).select(*select_cols(metric_col)).limit(top_k)

def to_table_pdf(df_spark, metric_col, title):
    pdf = df_spark.toPandas()
    # styled table
    styler = pdf.style.background_gradient(subset=[metric_col], cmap="YlGn")
    try:
        styler = styler.hide_index()
    except Exception:
        styler = styler.hide(axis="index")
    display(styler.set_caption(title))
    # bar chart
    plt.figure(figsize=(10,4))
    plt.bar(pdf.iloc[:10][pdf.columns[1]], pdf.iloc[:10][metric_col], color="#4CAF50")
    plt.xticks(rotation=45, ha="right")
    plt.title(title + " (Top 10 bar)")
    plt.ylabel(metric_col)
    plt.tight_layout()
    plt.show()
    return pdf

In [ ]:
# Compute and display for selected metric
metric_map = {m: (kcal_col, g_col) for (m, kcal_col, g_col) in densities}
if display_metric not in metric_map:
    print("Selected display_metric not available:", display_metric)
else:
    m_kcal, m_100g = metric_map[display_metric][0], metric_map[display_metric][1]
    top_kcal_df = topk(df_filt, m_kcal, desc=order_desc)
    top_100g_df = topk(df_filt, m_100g, desc=order_desc)
    print("Showing Top-K for metric:", display_metric)
    pdf_kcal = to_table_pdf(top_kcal_df, m_kcal, f"Top-{top_k} {display_metric} per kcal")
    pdf_100g = to_table_pdf(top_100g_df, m_100g, f"Top-{top_k} {display_metric} per 100g")
    if save_outputs:
        base = f"{output_base}/{display_metric}"
        (top_kcal_df
         .coalesce(1)
         .write.mode("overwrite").option("header", True).csv(f"{base}/per_kcal_csv"))
        (top_kcal_df
         .write.mode("overwrite").parquet(f"{base}/per_kcal_parquet"))
        (top_100g_df
         .coalesce(1)
         .write.mode("overwrite").option("header", True).csv(f"{base}/per_100g_csv"))
        (top_100g_df
         .write.mode("overwrite").parquet(f"{base}/per_100g_parquet"))
        print("Saved outputs to:", base)

In [ ]:
# Batch save all metrics (optional)
if save_outputs:
    for m, m_kcal, m_100g in densities:
        base = f"{output_base}/{m}"
        top_kcal_df = topk(df_filt, m_kcal, desc=order_desc)
        top_100g_df = topk(df_filt, m_100g, desc=order_desc)
        (top_kcal_df
         .coalesce(1)
         .write.mode("overwrite").option("header", True).csv(f"{base}/per_kcal_csv"))
        (top_kcal_df
         .write.mode("overwrite").parquet(f"{base}/per_kcal_parquet"))
        (top_100g_df
         .coalesce(1)
         .write.mode("overwrite").option("header", True).csv(f"{base}/per_100g_csv"))
        (top_100g_df
         .write.mode("overwrite").parquet(f"{base}/per_100g_parquet"))
    print("Saved batch outputs under:", output_base)